In [ ]:
!nvidia-smi
!python -V

Sat May 16 12:58:29 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
!pip -q install -U pip
!pip -q install -U "bitsandbytes>=0.46.1" "transformers>=4.41" "accelerate>=0.30"

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 39.6 MB/s eta 0:00:00



#### Restart Session

In [ ]:
!pip -q install fastapi "uvicorn[standard]" pillow pyngrok

In [ ]:
import bitsandbytes as bnb
print("bitsandbytes version:", bnb.__version__)

bitsandbytes version: 0.49.2


In [ ]:
!python -V
!pip show bitsandbytes

Python 3.12.13
Name: bitsandbytes
Version: 0.49.2
Summary: k-bit optimizers and matrix multiplication routines.
Home-page: https://github.com/bitsandbytes-foundation/bitsandbytes
Author: 
Author-email: Tim Dettmers <dettmers@cs.washington.edu>
License-Expression: MIT
Location: /usr/local/lib/python3.12/dist-packages
Requires: numpy, packaging, torch
Required-by: 


In [ ]:
import os
from google.colab import userdata

# Retrieve secrets and set environment variables
os.environ["HF_TOKEN"] = userdata.get('HF_TOKEN')
os.environ["NGROK_AUTHTOKEN"] = userdata.get('NGROK_AUTHTOKEN')

print("Tokens successfully loaded into environment!")

Tokens successfully loaded into environment!


In [ ]:
import re
import json

In [ ]:
def extract_json(text: str):
    if not text:
        return None

    # Strip ```json fences if present
    cleaned = re.sub(r"^```json\\s*|```$", "", text.strip(), flags=re.IGNORECASE | re.MULTILINE)

    # If still not valid JSON, pull the first {...} block
    match = re.search(r"\\{.*\\}", cleaned, flags=re.DOTALL)
    if match:
        cleaned = match.group(0)

    try:
        return json.loads(cleaned)
    except Exception:
        return None

In [ ]:
import os
os.environ["USE_4BIT"] = "0"

In [ ]:
import base64
import hashlib
import io
import json
import os
import re
import threading
import uuid
import torch
from PIL import Image
from fastapi import FastAPI
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel, Field
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig

try:
    from transformers import AutoModelForImageTextToText
except Exception:
    AutoModelForImageTextToText = None

try:
    from transformers import AutoModelForVision2Seq
except Exception:
    AutoModelForVision2Seq = None

MODEL_ID = "google/gemma-4-E2B-it"
DEBUG_IMAGES = os.environ.get("DEBUG_IMAGES", "0") == "1"
USE_4BIT = os.environ.get("USE_4BIT", "1") == "1"

processor = AutoProcessor.from_pretrained(
    MODEL_ID,
    token=os.environ["HF_TOKEN"],
    trust_remote_code=True,
    )

torch_dtype = torch.float16

if USE_4BIT:
    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16,
    )
else:
    bnb_config = None

model_kwargs = {
    "device_map": "auto",
    "torch_dtype": torch_dtype,
    "token": os.environ["HF_TOKEN"],
    "trust_remote_code": True,
}
if bnb_config is not None:
    model_kwargs["quantization_config"] = bnb_config


def load_model(model_id: str, kwargs: dict):
    candidates = []
    if AutoModelForImageTextToText is not None:
        candidates.append(AutoModelForImageTextToText)
    if AutoModelForVision2Seq is not None:
        candidates.append(AutoModelForVision2Seq)
    candidates.append(AutoModelForCausalLM)

    errors = []
    for cls in candidates:
        try:
            model = cls.from_pretrained(model_id, **kwargs)
            return model, cls.__name__
        except Exception as exc:
            errors.append(f"{cls.__name__}: {type(exc).__name__}")

    raise RuntimeError("Model load failed. Tried: " + ", ".join(errors))


model, MODEL_CLASS_NAME = load_model(MODEL_ID, model_kwargs)
model.eval()
inference_lock = threading.Lock()

if DEBUG_IMAGES:
    model_type = getattr(model.config, "model_type", None)
    has_vision = hasattr(model.config, "vision_config") or hasattr(model.config, "image_token_index")
    print("model_class:", MODEL_CLASS_NAME)
    print("model_type:", model_type)
    print("has_vision_config:", bool(has_vision))


class InferenceRequest(BaseModel):
    request_id: str | None = None
    prompt: str = Field(default="")
    frames: list[str]


def ensure_request_id(request_id: str | None) -> str:
    return request_id or f"req-{uuid.uuid4()}"


def decode_image(data_url: str):
    if not data_url:
        return None, {"error": "empty_data_url"} if DEBUG_IMAGES else None

    raw_data = data_url
    if "," in data_url:
        data_url = data_url.split(",", 1)[1]

    try:
        img_bytes = base64.b64decode(data_url)
    except Exception as exc:
        return None, {"error": f"base64_decode_failed: {exc}"} if DEBUG_IMAGES else None

    try:
        image = Image.open(io.BytesIO(img_bytes)).convert("RGB")
    except Exception as exc:
        return None, {"error": f"image_decode_failed: {exc}"} if DEBUG_IMAGES else None

    debug = None
    if DEBUG_IMAGES:
        debug = {
            "size": [image.size[0], image.size[1]],
            "mode": image.mode,
            "data_url_prefix": raw_data[:40],
            "data_url_md5": hashlib.md5(raw_data.encode("utf-8")).hexdigest(),
        }

    return image, debug


def parse_json_response(text: str):
    if not text:
        return None

    cleaned = re.sub(r"^```json\s*|```$", "", text.strip(), flags=re.IGNORECASE | re.MULTILINE)
    matches = re.findall(r"\{[\s\S]*?\}", cleaned)

    for candidate in matches:
        try:
            return json.loads(candidate)
        except Exception:
            continue

    return None


def normalize_response(data: dict, request_id: str, image_debug: list | None = None):
    location = data.get("location") if isinstance(data.get("location"), str) else ""
    location_match = re.search(r"-?\d+(?:\.\d+)?\s*,\s*-?\d+(?:\.\d+)?", location or "")
    risk_level = data.get("risk_level") if data.get("risk_level") in {"High", "Medium", "Low"} else "High"
    confidence_value = data.get("confidence")
    try:
        confidence_value = float(confidence_value)
    except Exception:
        confidence_value = 0.5
    confidence_value = max(0.0, min(1.0, confidence_value))

    recommended_actions = data.get("recommended_actions")
    if not isinstance(recommended_actions, list):
        recommended_actions = ["Manual review recommended"]

    response = {
        "request_id": request_id,
        "location": location_match.group(0) if location_match else "16.5062, 80.6480",
        "risk_level": risk_level,
        "help_needed": data.get("help_needed") if isinstance(data.get("help_needed"), str) else "Analysis Error",
        "description": data.get("description") if isinstance(data.get("description"), str) else "No description provided.",
        "confidence": confidence_value,
        "recommended_actions": recommended_actions,
    }
    if DEBUG_IMAGES:
        response["debug_images"] = image_debug or []
    return response


def run_model(prompt: str, frames: list[str], request_id: str):
    decoded = [decode_image(f) for f in frames[:3]]
    images = [item[0] for item in decoded if item and item[0] is not None]
    image_debug = [item[1] for item in decoded if item and item[1] is not None]

    if not images:
        return normalize_response({
            "risk_level": "High",
            "help_needed": "Analysis Error",
            "description": "No valid images were received by the backend.",
            "confidence": 0.0,
            "recommended_actions": ["Retry image upload", "Verify backend connection"],
        }, request_id, image_debug)

    user_prompt = prompt or "Assess the image and return emergency response JSON."
    system_prompt = (
        "You are a disaster response vision analyst. "
        "Return ONLY raw JSON with keys: request_id, location, risk_level, help_needed, description, confidence, recommended_actions. "
        "Do NOT return markdown, code blocks, or extra text. "
        "risk_level must be High, Medium, or Low. confidence must be 0..1."
    )
    user_text = f"{user_prompt}\nEcho request_id exactly as: {request_id}."

    image_placeholders = [{"type": "image"} for _ in images]
    messages = [
        {"role": "system", "content": system_prompt},
        {"role": "user", "content": [*image_placeholders, {"type": "text", "text": user_text}]},
    ]

    text = processor.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=False,
    )

    inputs = processor(
        text=text,
        images=images,
        return_tensors="pt",
        padding=True,
    ).to(model.device)

    pixel_values = inputs.get("pixel_values")
    if pixel_values is None:
        return normalize_response({
            "risk_level": "High",
            "help_needed": "Analysis Error",
            "description": "Image tensor missing from model inputs. Check model class or quantization.",
            "confidence": 0.0,
            "recommended_actions": ["Switch off 4-bit", "Recreate model/processor"],
        }, request_id, image_debug)

    with inference_lock:
        with torch.inference_mode():
            outputs = model.generate(
                **inputs,
                max_new_tokens=384,
                do_sample=False,
            )

    response = processor.decode(
        outputs[0][inputs["input_ids"].shape[-1]:],
        skip_special_tokens=True,
    )

    parsed = parse_json_response(response)
    if parsed:
        parsed["request_id"] = request_id
        return normalize_response(parsed, request_id, image_debug)

    return normalize_response({
        "risk_level": "High",
        "help_needed": "Analysis Error",
        "description": "Model output could not be parsed into JSON.",
        "confidence": 0.5,
        "recommended_actions": ["Manual review recommended"],
    }, request_id, image_debug)


app = FastAPI()

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=False,
    allow_methods=["*"],
    allow_headers=["*"],
)

@app.post("/infer")
def infer(req: InferenceRequest):
    request_id = ensure_request_id(req.request_id)
    return run_model(req.prompt, req.frames, request_id)

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

[transformers] `torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/10.2G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1951 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

In [ ]:
#!fuser -k 8000/tcp # for killling any pre backends are running before starting the main server

In [ ]:
import threading
import uvicorn
from pyngrok import ngrok

# Start FastAPI server in background
threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000),
    daemon=True
).start()

# Start ngrok tunnel
ngrok.set_auth_token(os.environ["NGROK_AUTHTOKEN"])
public_url = ngrok.connect(8000).public_url
print("Public URL:", public_url)

INFO:     Started server process [8356]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)


PyngrokNgrokHTTPError: ngrok client exception, API returned 502: {"error_code":103,"status_code":502,"msg":"failed to start tunnel","details":{"err":"failed to start tunnel: The endpoint 'https://unhurting-nonmediative-deegan.ngrok-free.dev' is already online. Either\n1. stop your existing endpoint first, or\n2. start both endpoints with `--pooling-enabled` to load balance between them.\r\n\r\nERR_NGROK_334\r\n"}}


In [ ]:

import base64
import io
from PIL import Image

img = Image.new("RGB", (128, 128), color="gray")
buf = io.BytesIO()
img.save(buf, format="JPEG")
b64 = "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()


In [ ]:
fire_url = '/content/fire_1.jpg'
img = Image.open(fire_url)
buf = io.BytesIO()
img.save(buf, format="JPEG")
b64 = "data:image/jpeg;base64," + base64.b64encode(buf.getvalue()).decode()

In [ ]:
import requests
import json
import uuid

request_id = f"req-{uuid.uuid4()}"
payload = {
    "request_id": request_id,
    "prompt": "Assess the disaster image and return JSON.",
    "frames": [b64],
}
resp = requests.post(
    "http://localhost:8000/infer",
    json=payload,
)
print("status:", resp.status_code)
print("text:", resp.text[:500])

INFO:     127.0.0.1:33706 - "POST /infer HTTP/1.1" 200 OK
status: 200
text: {"request_id":"req-b255eb19-88b0-4a20-a0ae-048e29a1459d","location":"16.5062, 80.6480","risk_level":"High","help_needed":"Immediate rescue and fire suppression","description":"A residential building is engulfed in intense, large-scale fire, with significant structural damage visible. Smoke and flames are highly visible, indicating an active and severe fire event.","confidence":0.98,"recommended_actions":["Activate emergency services immediately (Fire Department, EMS).","Establish a safe perimete


In [ ]:
import requests
import json
import uuid

request_id = f"req-{uuid.uuid4()}"
payload = {
    "request_id": request_id,
    "prompt": "Assess the disaster image and return JSON.",
    "frames": [b64],
}
resp = requests.post(
    "https://unhurting-nonmediative-deegan.ngrok-free.dev/infer",
    json=payload,
)
print("status:", resp.status_code)
print("text:", resp.text[:500])

INFO:     34.142.199.245:0 - "POST /infer HTTP/1.1" 200 OK
status: 200
text: {"request_id":"req-f7718621-2295-4890-b593-9b79536b690d","location":"16.5062, 80.6480","risk_level":"High","help_needed":"Immediate rescue and fire suppression","description":"A residential building is engulfed in intense, large-scale fire, with flames consuming the roof and exterior walls. Smoke is visible, and debris is present. A person is visible near the structure, possibly attempting to assist or escape.","confidence":0.98,"recommended_actions":["Activate emergency services immediately (Fi


In [ ]:
print("text:", resp.text[:2000])

text: {"request_id":"req-f7718621-2295-4890-b593-9b79536b690d","location":"16.5062, 80.6480","risk_level":"High","help_needed":"Immediate rescue and fire suppression","description":"A residential building is engulfed in intense, large-scale fire, with flames consuming the roof and exterior walls. Smoke is visible, and debris is present. A person is visible near the structure, possibly attempting to assist or escape.","confidence":0.98,"recommended_actions":["Activate emergency services immediately (Fire Department, EMS).","Establish a safe perimeter to protect the public.","Coordinate aerial support if necessary.","Monitor for secondary hazards (structural collapse, utility failure)."]}


In [ ]:
# from pyngrok import ngrok

# try:
#     ngrok.kill()
# except Exception:
#     pass